## Data preprocessing

In [78]:
# Imports et fonctions
import pandas as pd
import xml.etree.ElementTree as ET
import ast
import re


def parse_orphanet_xml(xml_file):
    # Charger et parser le fichier
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    results = []

    # On itère sur chaque balise <Disorder>
    for disorder in root.findall('.//Disorder'):
        orpha_code = disorder.find('OrphaCode').text
        # On spécifie lang='en' car il peut y avoir plusieurs balises Name
        disorder_name = disorder.find("./Name[@lang='en']").text
        
        # Un trouble peut avoir plusieurs gènes associés
        gene_associations = disorder.findall('.//DisorderGeneAssociation')
        
        for association in gene_associations:
            gene = association.find('Gene')
            if gene is not None:
                gene_symbol = gene.find('Symbol').text
                gene_code = gene.get("id")
                
                # Stockage des données
                results.append({
                    'OrphaCode': orpha_code,
                    'DiseaseName': disorder_name,
                    'GeneSymbol': gene_symbol,
                    'GeneId': gene_code
                })
                
    return results

In [79]:



# --- Utilisation ---
# Remplacez 'votre_fichier.xml' par le nom de votre fichier
data = parse_orphanet_xml('genes_associated_with_rare_diseases.xml')

# Conversion en DataFrame pour manipuler les données facilement
mappingGTD = pd.DataFrame(data) # Genes To Diseases


# Exemple pour sauvegarder en CSV
mappingGTD.to_csv('mapping_genes_maladies_orphanet_13_05_2026.csv', index=False)

# Données Patients eHOP
patData = pd.read_csv("../../../data/raw_data/clinical_side/ehop_one_gene_export_with_keys_without_na.tsv", sep="\t",)

rdPatData = patData[["ID_PAT_ETUDE", "IPP", "IPP_clef", "key_for_chaining", "gene_symbol",	"gene_hgnc_id"]].dropna()
rdPatData["gene_hgnc_id"] = rdPatData["gene_hgnc_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Merging 
#Same writing
rdPatData['gene_symbol'] = rdPatData['gene_symbol'].astype(str).str.strip().str.upper()
mappingGTD['GeneSymbol'] = mappingGTD['GeneSymbol'].astype(str).str.strip().str.upper()

# Merging
dfMerged = pd.merge(
    rdPatData,
    mappingGTD,
    left_on="gene_symbol",
    right_on="GeneSymbol",
    how="left"
)

# print(dfMerged.head())
print(dfMerged.shape)

# Identification of patients with multiple diseases possibility
counts = dfMerged.groupby("ID_PAT_ETUDE")["OrphaCode"].nunique()
patients_multi_diseases = counts[counts > 1].index

# Logs
log_data = dfMerged[dfMerged["ID_PAT_ETUDE"].isin(patients_multi_diseases)]
log_data = log_data.sort_values(by="ID_PAT_ETUDE")

# Ajouter une colonne qui indique combien de maladies ce patient a au total
dfMerged['NB_DISEASES_TOTAL'] = dfMerged.groupby('ID_PAT_ETUDE')['OrphaCode'].transform('nunique')

# Filtrer le log pour n'avoir que les cas complexes
log_data = dfMerged[dfMerged['NB_DISEASES_TOTAL'] > 1].sort_values(by=['NB_DISEASES_TOTAL', 'ID_PAT_ETUDE'], ascending=False)


# Données termes HPO
hpoData = pd.read_csv("../../../data/modified_data/for_phenoBERT/full_table_agg_patient_level_phenobert03032025.csv", sep=",")
# + uniquement colonnes hpo et ID depuis le fichier HPO
colonnes_hpo = ["PATIENT_ID", "hpo_unique_list_name"] 
hpoData= log_data.merge(
    hpoData[colonnes_hpo],
    left_on="ID_PAT_ETUDE",   # clé dans patient_multi_disease
    right_on="PATIENT_ID",    # clé dans le fichier HPO
    how="left"                # garde TOUS les patients, même sans HPO
)

# La colonne PATIENT_ID est redondante après la jointure, on peut la supprimer
hpoData = hpoData.drop(columns=["PATIENT_ID"])


print(hpoData.head())

# Saving
hpoData.to_csv("patients_multi_diseases.csv", index=False)

(1120, 10)
                               ID_PAT_ETUDE      IPP  IPP_clef  \
0  0F1E15F1331F803A28C1DDFF8598195DAE341562  1399038  13990380   
1  0F1E15F1331F803A28C1DDFF8598195DAE341562  1399038  13990380   
2  0F1E15F1331F803A28C1DDFF8598195DAE341562  1399038  13990380   
3  0F1E15F1331F803A28C1DDFF8598195DAE341562  1399038  13990380   
4  0F1E15F1331F803A28C1DDFF8598195DAE341562  1399038  13990380   

   key_for_chaining gene_symbol gene_hgnc_id OrphaCode  \
0          13990380      COL2A1         2200    166100   
1          13990380      COL2A1         2200      2380   
2          13990380      COL2A1         2200    137678   
3          13990380      COL2A1         2200       485   
4          13990380      COL2A1         2200      1856   

                                         DiseaseName GeneSymbol GeneId  \
0  Autosomal dominant otospondylomegaepiphyseal d...     COL2A1  15769   
1                         Legg-Calvé-Perthes disease     COL2A1  15769   
2  Spondyloepiphyseal